# Quadrant Classification: CRISPR vs RNA-seq (R Version)

This notebook uses R to plot a quadrant chart:
- X-axis: RNA-seq |log2FC| (absolute value)
- Y-axis: CRISPR Score (gene-level log2FC or other scores; higher values indicate stronger enrichment)

Tasks:
- Clearly label the position of `ANPEP` (representative of high Y, low X)
- Highlight genes in the top-right quadrant (strong CRISPR enrichment + significant expression change) and export the gene list

In [ ]:
# Install required packages (uncomment on first run)
# install.packages(c("ggplot2", "dplyr", "readr", "ggrepel", "scales"))

library(ggplot2)
library(dplyr)
library(readr)
library(ggrepel)  # For avoiding label overlap
library(scales)

cat("✓ Packages loaded\n")

In [ ]:
# === Parameter Settings ===
# Input file path
INPUT_FILE <- "/path/to/CRISPR/results/crispr_rnaseq_merged.csv"

# Column name mapping
COL_GENE_ID <- "gene_id"
COL_SYMBOL <- "gene_symbol"
COL_CRISPR <- "crispr_score"   # Y-axis
COL_RNA_L2FC <- "rna_log2fc"   # X-axis (absolute value)
COL_RNA_PADJ <- "rna_padj"     # Optional

# Threshold settings
Y_THRESHOLD <- 2.0    # CRISPR strong enrichment threshold
X_THRESHOLD <- 1.0    # Significant expression change threshold
PADJ_THRESHOLD <- 0.05  # Expression significance FDR

# Highlighted genes
HIGHLIGHT_GENES <- c("ANPEP", "WDR13", "LPP")

# Output directory
RESULTS_DIR <- "/path/to/CRISPR/results"

cat("✓ Parameters set\n")

In [ ]:
# === Data Loading and Preprocessing ===
cat("Loading data...\n")
if (!file.exists(INPUT_FILE)) {
  stop(paste("Input file not found:", INPUT_FILE))
}

# Read data
df <- read_csv(INPUT_FILE, show_col_types = FALSE)
cat("Loaded data shape:", nrow(df), "x", ncol(df), "\n")
cat("Columns:", paste(colnames(df), collapse = ", "), "\n")

# Check required columns
required_cols <- c(COL_SYMBOL, COL_CRISPR, COL_RNA_L2FC)
missing_cols <- setdiff(required_cols, colnames(df))
if (length(missing_cols) > 0) {
  stop(paste("Missing required columns:", paste(missing_cols, collapse = ", ")))
}

# Data preprocessing
df <- df %>%
  mutate(
    !!COL_CRISPR := as.numeric(!!sym(COL_CRISPR)),
    !!COL_RNA_L2FC := as.numeric(!!sym(COL_RNA_L2FC)),
    rna_log2fc_abs = abs(!!sym(COL_RNA_L2FC))
  ) %>%
  filter(!is.na(!!sym(COL_CRISPR)), !is.na(rna_log2fc_abs))

# RNA significance
if (COL_RNA_PADJ %in% colnames(df)) {
  df <- df %>% mutate(rna_sig = !!sym(COL_RNA_PADJ) < PADJ_THRESHOLD)
} else {
  df <- df %>% mutate(rna_sig = TRUE)
}

# Quadrant classification
df <- df %>%
  mutate(
    top_right = (!!sym(COL_CRISPR) >= Y_THRESHOLD) & 
                (rna_log2fc_abs >= X_THRESHOLD) & 
                rna_sig,
    is_highlight = toupper(!!sym(COL_SYMBOL)) %in% toupper(HIGHLIGHT_GENES)
  )

cat("Usable rows:", nrow(df), "\n")
cat("Top-right genes:", sum(df$top_right), "\n")

# Print highlighted gene information
cat("\nHighlighted gene information:\n")
for (gene in HIGHLIGHT_GENES) {
  gene_data <- df %>% filter(toupper(!!sym(COL_SYMBOL)) == toupper(gene))
  if (nrow(gene_data) > 0) {
    r <- gene_data[1]
    category <- if ("crispr_category" %in% colnames(r)) r$crispr_category else "Unknown"
    cat(sprintf("  %s: CRISPR=%.2f, RNA-log2FC=%.2f, |log2FC|=%.2f, Category=%s\n", 
                gene, r[[COL_CRISPR]], r[[COL_RNA_L2FC]], r$rna_log2fc_abs, category))
  } else {
    cat(sprintf("  %s: Not found\n", gene))
  }
}

head(df)

In [ ]:
# === Color Definitions ===
# CRISPR category colors (blue-red-orange-yellow color scheme)
crispr_colors <- c(
  "Depleted" = "#4472C4",              # Blue (Depleted)
  "Slightly Enriched" = "#FFD700",     # Yellow (Slightly Enriched)
  "Moderately Enriched" = "#FF8C00",  # Orange (Moderately Enriched)
  "Highly Enriched" = "#DC143C"       # Red (Highly Enriched)
)

# Note: Highlighted genes (ANPEP, WDR13, LPP) are distinguished by a thick black border;
# their fill color still uses their CRISPR category color

cat("✓ Color definitions complete\n")

In [ ]:
# === Draw Scatter Plot ===
cat("Creating plot...\n")

# Base scatter plot
p <- ggplot(df, aes(x = rna_log2fc_abs, y = !!sym(COL_CRISPR))) +
  # Background points (larger size, lower transparency, black border)
  {if ("crispr_category" %in% colnames(df)) {
    geom_point(aes(color = crispr_category, fill = crispr_category), 
               alpha = 0.7, size = 5.0, stroke = 0.5, color = "black", shape = 21)
  } else {
    geom_point(fill = "#bdbdbd", alpha = 0.4, size = 5.0, stroke = 0.5, color = "black", shape = 21)
  }} +
  
  # Top-right quadrant points (thicker border)
  {if (sum(df$top_right) > 0) {
    if ("crispr_category" %in% colnames(df)) {
      geom_point(data = df %>% filter(top_right), 
                 aes(fill = crispr_category),
                 alpha = 0.8, size = 5.0, stroke = 1.2, color = "black", shape = 21)
    } else {
      geom_point(data = df %>% filter(top_right),
                 fill = "#bdbdbd", alpha = 0.5, size = 5.0, stroke = 1.2, color = "black", shape = 21)
    }
  }} +
  
  # Highlighted gene points (ANPEP and WDR13) - using their group color
  {if (sum(df$is_highlight) > 0) {
    if ("crispr_category" %in% colnames(df)) {
      geom_point(data = df %>% filter(is_highlight),
                 aes(fill = crispr_category),
                 alpha = 0.9, size = 6.0, stroke = 2.0, color = "black", shape = 21)
    } else {
      geom_point(data = df %>% filter(is_highlight),
                 fill = "#bdbdbd", alpha = 0.9, size = 6.0, stroke = 2.0, color = "black", shape = 21)
    }
  }} +
  
  # Add gene labels (ANPEP and WDR13) - using black text
  {if (sum(df$is_highlight) > 0) {
    geom_text_repel(
      data = df %>% filter(is_highlight),
      aes(label = !!sym(COL_SYMBOL)),
      color = "black",
      size = 4.5,
      fontface = "bold",
      box.padding = 0.5,
      point.padding = 0.3,
      segment.color = "black",
      segment.size = 0.8,
      min.segment.length = 0.1,
      max.overlaps = Inf,
      nudge_x = 0.1,
      nudge_y = 0.5
    )
  }} +
  
  # Threshold lines
  geom_vline(xintercept = X_THRESHOLD, color = "gray", linetype = "dashed", 
             linewidth = 0.8, alpha = 0.7) +
  geom_hline(yintercept = Y_THRESHOLD, color = "gray", linetype = "dashed", 
             linewidth = 0.8, alpha = 0.7) +
  
  # Color settings
  {if ("crispr_category" %in% colnames(df)) {
    scale_fill_manual(values = crispr_colors, name = "CRISPR Category")
  }} +
  
  # Axis labels and title
  labs(
    x = "|RNA-seq log2FC|",
    y = "CRISPR Score (gene-level log2FC)",
    title = "Quadrant Classification: CRISPR vs RNA-seq"
  ) +
  
  # Theme settings (matching VolcanoPlot style)
  theme_bw(base_size = 13) +
  theme(
    text = element_text(face = "bold"),
    plot.title = element_text(size = 14, face = "bold", hjust = 0.5),
    panel.border = element_rect(colour = "black", fill = NA, linewidth = 1.2),
    axis.text = element_text(color = "black", face = "bold", size = 12),
    axis.title = element_text(color = "black", face = "bold", size = 14),
    axis.ticks.length = unit(0.25, "cm"),
    legend.position = "right",
    legend.title = element_text(size = 12, face = "bold"),
    legend.text = element_text(size = 11, face = "bold"),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  ) +
  
  # Uniform legend point border thickness
  guides(fill = guide_legend(
    override.aes = list(
      stroke = 0.8,  # Uniform border thickness
      size = 4,      # Uniform point size
      alpha = 0.8    # Uniform transparency
    )
  ))

# Display base plot
print(p)

In [ ]:
# === Save Results ===
cat("Saving results...\n")

# Create output directory
if (!dir.exists(RESULTS_DIR)) {
  dir.create(RESULTS_DIR, recursive = TRUE)
}

# Save figures (matching VolcanoPlot save method)
ggsave(
  filename = file.path(RESULTS_DIR, "Quadrant_Classification_R_scatter.pdf"),
  plot = p,
  device = cairo_pdf,  # Ensure editable fonts
  width = 10, height = 7, units = "in",
  dpi = 300
)

ggsave(
  filename = file.path(RESULTS_DIR, "Quadrant_Classification_R_scatter.png"),
  plot = p,
  width = 10, height = 7, units = "in",
  dpi = 300
)

cat("✓ Saved figures: Quadrant_Classification_R_scatter.png/pdf\n")

In [ ]:
# === Export Data ===
# Export top-right gene list
top_genes <- df %>% 
  filter(top_right) %>%
  select(all_of(c(COL_GENE_ID, COL_SYMBOL, COL_CRISPR, "rna_log2fc_abs")),
         any_of(c("crispr_category", "rna_deg_status", COL_RNA_L2FC))) %>%
  arrange(desc(!!sym(COL_CRISPR)))

out_csv <- file.path(RESULTS_DIR, "Quadrant_Classification_R_top_right_genes.csv")
write_csv(top_genes, out_csv)
cat("✓ Exported:", out_csv, "\n")

# Print top-right genes
cat(sprintf("\nTop-right quadrant genes (%d total):\n", nrow(top_genes)))
for (i in 1:nrow(top_genes)) {
  cat(sprintf("  - %s: CRISPR=%.2f, |RNA-log2FC|=%.2f\n", 
              top_genes[[COL_SYMBOL]][i], 
              top_genes[[COL_CRISPR]][i], 
              top_genes$rna_log2fc_abs[i]))
}

cat("\n✓ R version quadrant classification analysis complete!\n")

# Display top_genes table
head(top_genes, 10)